In [ ]:
# 1. What is Generative AI and what are its primary use cases across industries?

- Generative AI refers to a class of artifical intelligence models that can create new data such as text, images ,audio, code, and videos by learning patterns from existing data. Unlike traditional AI systems that only classify or predict , generative models can produce original content that resembles human-created data.

Primary use cases across industries:
1. Healthcare - Drug discovery, medical report generation , synthetic medical data creation, clinical documentation.
2. Finance - Automated report writing , fraud simulation, customer support chatbots, risk modeling.
3. Education - Personalized tutoring, question generation, automatic content creation.
4. Media & Entertainment - Story writing, script generation , music composition , video generation.
5. Software Development - Code generation , debugging assistance , documentation writing.
6. Marketing - Ad copy generation , email drafting , product description , social media content.
7. Manufacturing & Design - Product design generation, simulation data creation.

# 2. Explain the role of probabilistic modeling in generative models. How do these models differ from discriminative models?

- Probabilistics modeling in generative models involves learning the joint probability distribution P(X, Y) or P(X) of the data. These models capture how data is generated and can sample new data is generated and can sample new sample new data points from the learned distribution.

Role of probabilistic modeling :
. Represents uncertainty in data generation
. Enables sampling of new realistic data
. Learns latent representations of data

Differences between them are-

Aspects                           |                        Generative Models                           |                     Discriminative Models

What they learn                                          Joint distribution P(X,Y)orP(X)                                   Conditional distribution P(Y|X)

Purpose                                                  Generate new data and model data distribution                     Predict labels or outputs

Examples                                                 GANs, VAEs, HMMs                                                  Logistic Regression, SVM, CNN classifiers



# 3. What is the difference between Autoencoders and Variation Autoencoders (VAEs) in the context of text generation?

- An Autonoums (AE) is a neural network that compresses input data into a latent representation and reconstructs it back . it learns deterministic mappings.

- A Variational Autoencoder (VAE) is a probabilistic version of an autoencoder that learns a distribution over the latent space and allows sampling to generate new data.

Key differences:

Feature                                                           Autoencoder                                                    Variational Autoencoder

Latent space                                                     Fixed vector                                                    Probability distribution(means & variance)

Sampling                                                         Not suitable for generation                                     Can sample new data

Loss function                                                    Reconstruction loss only                                        Reconstructuion + KL divergence

Use in text generation                                           Limited                                                         Widely used for text and sentence generation



# 4. Describe the working of attention mechanisms in Neural Machine Transalation(NMT). Why are they critical?

- In Neural Machine Translation , attention mechanisms allow the decoder to focus on relevent parts of the soure sentence while generating each target word.

Working:

1. The encoder produces hidden states for each input word.
2. The decorder computes attention weights over all encoder states.
3. A context vector is created as a weighted sum of encoder states.
4. The decorder uses this context vector to predict the next word.

Why attention is critical:

. Handles long sentences better
. Improves translation accuracy
. Aligns source and target words dynamically
. Reduces information loss from fixed- length encoding

# 5. What ethical considerations must be addressed when using generative AI for creative content such as poetry or storytelling?

- 1. Plagiarism and Copyright - Generated content may copy existing works.
  2. Bias and Fairness - Models may reflect social, cultural , or gender biases.
  3. Misinformation - AI can generate misleading or harmful stories.
  4. Authencity - Difficulty in distinguishing human vs AI- created content.
  5. Ownership - Unclear who owns AI- generated creative works.
  6. Cultural Sensitivity - Risk of offensive or inappropriate content



In [ ]:
# 6. Train a simple VAE for text reconstruction?


import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Lambda
from tensorflow.keras import backend as K

# Dataset
sentences = ["The sky is blue ", "The sun is bright","The grass is green","The night is dark","The stars are shinning"]

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)
sequences = tokenizer.texts_to_sequences(sentences)
max_len = max(len(s) for s in sequences)
padded = pad_sequences(sequences, maxlen=max_len, padding='post')

vocab_size = len(tokenizer.word_index) + 1

# Simple VAE model
inputs = Input(shape=(max_len,))
h = Dense(16, activation='relu')(inputs)
z_mean = Dense(2)(h)
z_log_var = Dense(2)(h)

def sampling(args):
  z_mean, z_log_var = args
  epsilon = K.random_normal(shape=(K.shape(z_mean)[0],2))
  return z_mean + K.exp(0.5 * z_log_var) * epsilon

z = Lambda(sampling)([z_mean, z_log_var])
decoded = Dense(max_len, activation='sigmoid')(z)

vae = Model(inputs, decoded)
vae.compile(optimizer='adam', loss='mse')
vae.fit(padded, padded, epochs=50, verbose=0)

# Reconsrtuction
reconstructed = vae.predict(padded)
print("Original:", padded[0])
print("Reconstructed:", reconstructed[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
Original: [1 3 2 4]
Reconstructed: [0.7058027  0.8610281  0.83406293 0.1699931 ]


In [17]:
# 7. Translate an English paragraph into French and german using GPT?

from transformers import pipeline

translator_fr = pipeline("translation_en_to_fr")
translator_de = pipeline("translation_en_to_de")

text = "Artifical intelligence is transforming the world of technology."

fr = translator_fr(text)
de = translator_de(text)

print("Original:", text)
print("French:",fr[0]['translation_text'])
print("German:",de[0]['translation_text'])



No model was supplied, defaulted to google-t5/t5-base and revision a9723ea (https://huggingface.co/google-t5/t5-base).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu
No model was supplied, defaulted to google-t5/t5-base and revision a9723ea (https://huggingface.co/google-t5/t5-base).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


Original: Artifical intelligence is transforming the world of technology.
French: L'intelligence artificielle transforme le monde de la technologie.
German: Die künstlerische Intelligenz verändert die Welt der Technik.


In [22]:
# 8. Attention-based Encoder-Decoder for English-Spanish Translation?

import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Attention
from tensorflow.keras.models import Model
# Encoder
encoder_inputs = Input(shape=(None,64))
encoder_lstm = LSTM(128, return_state=True, return_sequences=True)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_inputs)
# Decoder with attention
attention = tf.keras.layers.Attention()([encoder_outputs, encoder_outputs])
decoder_inputs = Input(shape=(None,64))
decoder_lstm = LSTM(128, return_sequences=True)
decoder_outputs = decoder_lstm(decoder_inputs, initial_state=[state_h, state_c])

outputs = Dense(64, activation='softmax')(decoder_outputs)
model = Model([encoder_inputs, decoder_inputs], outputs)
model.compile(optimizer='adam', loss='categorical_crossentropy')

In [3]:
# 9. Poem Generation using GPT?

# Roses are red, violets are blue,
# Sugar is sweet, and so are you.
# The moon glows bright in silent skies,

from transformers import pipeline

generator = pipeline("text-generation", model="gpt2")
prompt = "Roses are red, violets are blue,"
poem = generator(prompt, max_length=40, num_return_sequences=1)

print(poem[0]['generated_text'])

Device set to use cpu
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=40) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Roses are red, violets are blue, and a few other insects have bright red leaves. (See Photos.)

You may have seen some of these animals that get a bit lost in the wild, but they were once among the most important predators of the insects, so they are a perfect example of the many, many different ways in which the insects can be used to attract other insects.

We've also learned that they are quite difficult to trap because they have to be kept in open areas, such as in a barn. (See Photos.)

We've also learned that some of the insects can grow to be quite aggressive in captivity if they are given the chance. The insects can eat as much as 10 pounds of meat, and they can eat as much as 4 pounds and more of meat if they are kept in a large enclosure.

And of course, if you're looking for a way to scare other insects away from the same area, they can be a great way to do that.

Dormant Insects

An insect is a small, black, or white ball of sticky green flesh. But it's not really a ball, i

In [4]:
# 10. Designing.......?


import random

# Sample plot templates
plots = [
    "A young hero discovers a hidden power and must save their kingdom.",
    "Two strangers meet during a storm and uncover a shared mysterious past.",
    "A scientist invents a machine that changes memories, but at a great cost.",
    "An ordinary student finds a portal to a magical world inside a library."
]

# Sample character traits
names = ["Arjun", "Maya", "Ravi", "Anaya", "Karan"]
traits = ["brave", "curious", "quiet", "ambitious", "kind"]
professions = ["writer", "scientist", "warrior", "student", "detective"]

def generate_story():
    plot = random.choice(plots)
    name = random.choice(names)
    trait = random.choice(traits)
    profession = random.choice(professions)

    character = f"{name} is a {trait} {profession} who dreams of changing the world."

    return plot, character

# Generate one sample story idea
plot, character = generate_story()

print("Generated Plot:")
print(plot)
print("\nGenerated Character Description:")
print(character)

Generated Plot:
A young hero discovers a hidden power and must save their kingdom.

Generated Character Description:
Karan is a ambitious detective who dreams of changing the world.
